# 01 — Data Preparation: Loading WIKI Metadata

## Syfte
Detta notebook-steg motsvarar kursens krav på **dataförberedelse**: vi läser in
råmetadatan för WIKI-delen av IMDB-WIKI-datasetet och verifierar att den är
strukturellt korrekt innan någon rensning eller transformation sker.

Vi separerar medvetet:
- **Inläsning** (`src/data_loader.py`) — läser `wiki.mat` oförändrat.
- **Rensning** (`src/preprocessing.py`, nästa steg) — hanterar saknade
  värden, konverterar MATLAB-datum till riktiga åldrar, m.m.

Detta gör att vi alltid kan gå tillbaka till rådatan om ett rensningssteg
visar sig vara felaktigt.

In [1]:
import sys
from pathlib import Path

# Add project root (parent of notebooks/) to sys.path so `src` is importable
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## Läs in rå-metadata

`load_wiki_mat` läser MATLAB-structen `wiki.mat` och returnerar en
DataFrame med en rad per ansiktsbild, med de råa fälten oförändrade
(ingen imputation, ingen typkonvertering av datum).

In [2]:
from src.data_loader import load_wiki_mat

df = load_wiki_mat("../data/raw/wiki_crop/wiki.mat")
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 62328 entries, 0 to 62327
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   full_path          62328 non-null  str    
 1   dob_matlab         62328 non-null  int32  
 2   photo_taken        62328 non-null  uint16 
 3   gender             59685 non-null  float64
 4   face_score         62328 non-null  float64
 5   second_face_score  4096 non-null   float64
 6   name               62204 non-null  str    
 7   face_location      62328 non-null  object 
dtypes: float64(3), int32(1), object(1), str(2), uint16(1)
memory usage: 3.2+ MB


### Kommentar till `df.head()`

Varje rad representerar en bild. `full_path` pekar på filen relativt
`wiki_crop/`, `dob_matlab` är ett MATLAB-serienummer för födelsedatum
(inte en läsbar datum ännu), och `face_location` är en boundingbox
`[x1, y1, x2, y2]` som markerar var ansiktet detekterades i originalbilden.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 62328 entries, 0 to 62327
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   full_path          62328 non-null  str    
 1   dob_matlab         62328 non-null  int32  
 2   photo_taken        62328 non-null  uint16 
 3   gender             59685 non-null  float64
 4   face_score         62328 non-null  float64
 5   second_face_score  4096 non-null   float64
 6   name               62204 non-null  str    
 7   face_location      62328 non-null  object 
dtypes: float64(3), int32(1), object(1), str(2), uint16(1)
memory usage: 3.2+ MB


### Kommentar till `df.info()`

- **62 328 rader** — matchar det kända, dokumenterade antalet bilder i
  WIKI-delen av datasetet. Bekräftar att hela `wiki.mat` lästes in korrekt.
- **`gender`: 59 685 / 62 328 icke-null** → cirka 2 643 saknade värden
  (~4,2 %). Detta är äkta, naturligt förekommande missing data i källan —
  inte något vi har introducerat — och blir ett konkret exempel att hantera
  under **dataförberedelse**.
- **`second_face_score`: endast 4 096 icke-null** — detta är *förväntat*
  och inte ett dataproblem: fältet är bara ifyllt när ansiktsdetektorn
  hittade ett *andra* ansikte i bilden. `NaN` betyder här "inget andra
  ansikte hittades", vilket är semantiskt annorlunda än vanlig saknad data
  och kommer hanteras därefter (t.ex. `NaN` → 0 eller en boolesk
  "has_second_face"-flagga, snarare än imputation).
- **`face_score`: 62 328 icke-null**, men detta fält innehåller sannolikt
  `-inf` för bilder där ingen ansikte alls detekterades. Det är alltså inte
  riktig "brist" i pandas mening, men kommer behöva filtreras/hanteras
  separat i preprocessing.
- **`name`: 62 204 / 62 328 icke-null** — ett litet antal saknade
  celebritetsnamn, ytterligare ett äkta missing-values-exempel.

**Slutsats:** Datasetet har flera typer av äkta, verkliga databrister
(saknad `gender`, saknat `name`, semantiskt "saknad" `second_face_score`,
och sannolikt `-inf`-värden i `face_score`) — vilket uppfyller kursens krav
på att hantera saknade värden med *naturligt förekommande* data, utan att
vi behöver simulera brus.